# Stage A3 — Linear stitching

**Experiment A — Representation Convergence (GPT-2 vs Pythia-160M)**

Hypothesis: *well-trained independent models share representations up to a
linear transformation.* The models were chosen from different labs,
architectures, tokenizers, and training corpora (OpenAI/WebText vs
EleutherAI/The Pile), both trained from scratch — so any alignment found
must have been discovered independently by each training run.


## What this stage does
The strong test. For each matched layer pair, fits **one ridge-regression
matrix** (closed form — the models themselves are never trained) mapping
GPT-2's representation into Pythia's, and measures held-out R².

The transformation class is the whole game:
- *Any* nonlinear translator → vacuous (can map anything to anything)
- *Identity* (no transform) → trivially fails (permutation symmetry)
- **Linear** → strong enough to undo rotations/permutations/scalings,
  too weak to create information that is not already there.

## Success criterion
Held-out R² > 0.7 in the middle layers (edges are tokenizer-specific and
always lower), and far above the random-baseline curve.


In [ ]:
# Storage setup — where stage artifacts (.npz, .png) are read/written.
# Each stage reads the previous stage's output from DATA_DIR.
#
# Option 1 (default): current directory. Works if you run ALL stages in
# the SAME runtime/session. In Colab, a new notebook = a new VM, so files
# from a previous notebook are gone.
#
# Option 2 (Colab, persistent): mount Google Drive and point DATA_DIR
# there — artifacts survive across notebooks and sessions:
#
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

import os
os.environ.setdefault("DATA_DIR", ".")
print("DATA_DIR =", os.path.abspath(os.environ["DATA_DIR"]))

In [ ]:
# Configuration and imports
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
DATA_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [ ]:
# Functions
def stitch_r2(X, Y, alpha=1.0):
    """Fit Y ~ X @ W with ridge; return held-out R^2 (variance-weighted)."""
    Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.25,
                                          random_state=0)
    # standardize inputs for stable ridge
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
    Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd
    reg = Ridge(alpha=alpha).fit(Xtr, Ytr)
    return r2_score(Yte, reg.predict(Xte),
                    multioutput="variance_weighted")


def matched_indices(nA, nB):
    """Match layers proportionally (models may differ in depth)."""
    return [(i, round(i * (nB - 1) / (nA - 1))) for i in range(nA)]


def main():
    data = np.load(str(DATA_DIR / "activations.npz"))
    A, B, R = data["A_layers"], data["B_layers"], data["R_layers"]

    pairs = matched_indices(A.shape[0], B.shape[0])
    r2_trained, r2_random = [], []

    for i, j in pairs:
        rt = stitch_r2(A[i], B[j])
        rr = stitch_r2(A[i], R[j])
        r2_trained.append(rt)
        r2_random.append(rr)
        print(f"GPT-2 L{i:2d} -> Pythia L{j:2d}:  "
              f"trained R2={rt:.3f}   random R2={rr:.3f}")

    layers = [i for i, _ in pairs]
    plt.figure(figsize=(9, 5))
    plt.plot(layers, r2_trained, "o-", label="A -> B (trained)")
    plt.plot(layers, r2_random, "s--", label="A -> random baseline")
    plt.axhline(0.7, color="gray", ls=":", label="success threshold")
    plt.xlabel("GPT-2 layer")
    plt.ylabel("Held-out R² of a single linear map")
    plt.title("Linear stitching: are the spaces isomorphic?")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(DATA_DIR / "stitching_r2.png"), dpi=150)

    print(f"\nMean R2 trained: {np.mean(r2_trained):.3f}")
    print(f"Mean R2 random:  {np.mean(r2_random):.3f}")
    print("Saved stitching_r2.png")
    print("Hypothesis supported if trained curve sits high (>0.7 mid-layers) "
          "and far above the random baseline.")

In [ ]:
# Run the stitching (requires activations.npz from stage A1)
main()